# Plotline land-change encoder: full-scale LeJEPA training on a Kaggle GPU

Plotline checks whether EUDR commodity plots were deforested after 31 December 2020. The verdict comes from
official forest maps (EU JRC GFC2020, Hansen Global Forest Change); a **tiny self-supervised encoder trained
with LeJEPA** (SIGReg, Balestriero & LeCun 2025) adds a second signal: the cosine distance between a plot's
2020 and current Sentinel-2 embeddings.

The repository trains a demo-scale encoder on CPU in minutes (`make train`, 9 regions). This notebook runs the
same code at full scale:

1. fetch 32 commodity landscapes (9 demo + 23 extra) from Microsoft Planetary Computer (anonymous), Hansen
   GFC-2025-v1.13 and JRC GFC2020 V4 — about 30-60 min with internet on;
2. train with LeJEPA on the GPU (20k steps, batch 512, 8 views);
3. export `encoder.npz` (batch norm folded, NumPy forward pass, < 2 MB) and check NumPy = PyTorch;
4. evaluate at plot level against Hansen loss on regions held out of training (ROC-AUC, precision, recall),
   with NDVI / NBR differencing and a random-init encoder as baselines.

**Settings:** Accelerator GPU (T4 or P100), Internet on. Copy `/kaggle/working/plotline/ml/artifacts/` back to
`ml/artifacts/` in the repository to use the new encoder in the API.

In [ ]:
%%bash
pip install -q \
    "lejepa @ git+https://github.com/galilai-group/lejepa.git@c293d291ca87cd4fddee9d3fffe4e914c7272052" \
    "rasterio>=1.4" "pystac-client>=0.8" "planetary-computer>=1.0" "scikit-learn>=1.5"
nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
import pathlib

ROOT = pathlib.Path("/kaggle/working/plotline")
(ROOT / "ml" / "plotline_ml").mkdir(parents=True, exist_ok=True)
(ROOT / "data").mkdir(exist_ok=True)
os.chdir(ROOT / "ml")
print(os.getcwd())

## Package sources (generated from `ml/plotline_ml/`)

In [ ]:
%%writefile /kaggle/working/plotline/ml/plotline_ml/__init__.py

In [ ]:
%%writefile /kaggle/working/plotline/ml/plotline_ml/regions.py
"""Study regions for demo-scale training and evaluation.

Each region is a 10.24 km x 10.24 km window (1024 x 1024 Sentinel-2 pixels at 10 m) in a commodity
landscape with known forest loss since 2021. Centres were chosen by sampling the Hansen Global Forest
Change loss-year layer; see docs/adr/0004-lejepa-change-signal.md.
"""

from __future__ import annotations

from dataclasses import dataclass


@dataclass(frozen=True)
class Region:
    key: str
    country: str  # ISO 3166-1 alpha-2
    commodity: str
    lat: float
    lon: float


REGIONS: tuple[Region, ...] = (
    Region("ci_cavally", "CI", "cocoa", 6.25, -7.85),
    Region("ci_soubre", "CI", "cocoa", 5.85, -6.55),
    Region("gh_western_north", "GH", "cocoa", 6.25, -2.75),
    Region("br_novo_progresso", "BR", "cattle", -7.05, -55.60),
    Region("br_porto_velho", "BR", "cattle", -9.35, -64.60),
    Region("id_riau", "ID", "oil_palm", 0.35, 101.85),
    Region("id_east_kalimantan", "ID", "oil_palm", 0.60, 116.40),
    Region("pe_san_martin", "PE", "coffee", -6.90, -76.60),
    Region("pe_madre_de_dios", "PE", "cocoa", -12.75, -69.90),
)

# Full-scale set for GPU training (Kaggle notebook): more commodity frontiers. Centres are approximate
# deforestation hotspots; check the printed loss share after fetching and drop windows without forest.
EXTRA_REGIONS: tuple[Region, ...] = (
    Region("br_sao_felix_xingu", "BR", "cattle", -6.60, -51.90),
    Region("br_colniza", "BR", "soy", -9.20, -59.00),
    Region("br_acre", "BR", "cattle", -9.90, -67.50),
    Region("br_labrea", "BR", "cattle", -7.90, -64.80),
    Region("bo_chiquitania", "BO", "soy", -16.50, -61.50),
    Region("py_boqueron", "PY", "cattle", -21.50, -60.50),
    Region("co_caqueta", "CO", "cattle", 1.00, -74.50),
    Region("pe_ucayali", "PE", "cocoa", -8.40, -74.60),
    Region("ci_mont_peko", "CI", "cocoa", 7.00, -7.30),
    Region("gh_ashanti", "GH", "cocoa", 6.60, -1.30),
    Region("lr_nimba", "LR", "rubber", 6.00, -9.30),
    Region("cm_east", "CM", "cocoa", 3.50, 13.50),
    Region("ng_cross_river", "NG", "cocoa", 6.00, 8.70),
    Region("cd_tshopo", "CD", "wood", 0.50, 25.00),
    Region("id_ketapang", "ID", "oil_palm", -1.80, 110.20),
    Region("id_jambi", "ID", "oil_palm", -1.60, 103.40),
    Region("id_central_kalimantan", "ID", "oil_palm", -2.20, 113.00),
    Region("id_merauke", "ID", "oil_palm", -7.90, 140.30),
    Region("my_sarawak", "MY", "oil_palm", 2.90, 113.00),
    Region("pg_madang", "PG", "oil_palm", -5.50, 145.50),
    Region("vn_central_highlands", "VN", "coffee", 12.70, 108.00),
    Region("la_bolaven", "LA", "coffee", 15.00, 106.00),
    Region("et_jimma", "ET", "coffee", 7.70, 36.80),
)
ALL_REGIONS: tuple[Region, ...] = REGIONS + EXTRA_REGIONS


def regions_for(name: str) -> tuple[Region, ...]:
    """``demo`` (9 regions, CPU in minutes) or ``full`` (all regions, GPU)."""
    return ALL_REGIONS if name == "full" else REGIONS


# Baseline imagery strictly before the EUDR cut-off (31 Dec 2020); current imagery after the last
# Hansen loss year in the release we use (GFC-2025-v1.13 maps loss up to and including 2025).
BASELINE_WINDOW = "2020-01-01/2020-12-31"
CURRENT_WINDOW = "2026-01-01/2026-10-31"

HANSEN_VERSION = "GFC-2025-v1.13"
HANSEN_BASE = f"https://storage.googleapis.com/earthenginepartners-hansen/{HANSEN_VERSION}"
JRC_GFC2020_COG = (
    "https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/FOREST/GFC2020/LATEST/single-cog/JRC_GFC2020_V4_COG.tif"
)
JRC_VERSION = "GFC2020-V4"
STAC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"

BANDS: tuple[str, ...] = ("B02", "B03", "B04", "B08", "B11")
WINDOW_PX = 1024
PATCH_PX = 32

In [ ]:
%%writefile /kaggle/working/plotline/ml/plotline_ml/fetch.py
"""Download demo-scale Earth-observation data for every study region (no account needed).

For each region and each epoch (2020 baseline, current) we:
1. search Sentinel-2 L2A on Microsoft Planetary Computer (anonymous SAS signing) for scenes on one
   MGRS tile, least cloudy first;
2. read a 1024 x 1024 window at 10 m for five bands plus the scene classification layer (SCL);
3. build a per-pixel median composite of clear observations (SCL vegetation / bare / water / unclassified);
and alongside it read, reprojected onto the same grid:
- Hansen Global Forest Change ``lossyear`` and ``treecover2000`` (public Google Cloud Storage);
- the EU JRC Global Forest Cover 2020 map (JRC open data COG).

Output: ``data/eo/regions/<key>.npz``. Re-running skips regions that already exist.

    python -m plotline_ml.fetch [--regions ci_cavally,gh_western_north] [--scenes 5]
"""

from __future__ import annotations

import argparse
import json
import math
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import planetary_computer
import pystac_client
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject, transform_bounds
from rasterio.windows import from_bounds

from plotline_ml.regions import (
    BANDS,
    BASELINE_WINDOW,
    CURRENT_WINDOW,
    HANSEN_BASE,
    HANSEN_VERSION,
    JRC_GFC2020_COG,
    JRC_VERSION,
    STAC_URL,
    WINDOW_PX,
    Region,
    regions_for,
)

ROOT = Path(__file__).resolve().parents[2]
OUT = ROOT / "data" / "eo" / "regions"
CLEAR_SCL = (4, 5, 6)  # vegetation, not vegetated, water (unclassified pixels are often cloud edges)
CLOUD_BLUE = 0.13  # B02 reflectance above this is haze/cloud over tropical land cover

_GDAL_ENV = {
    "GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR",
    "GDAL_HTTP_MAX_RETRY": "4",
    "GDAL_HTTP_RETRY_DELAY": "2",
    "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif,.TIF",
    "GDAL_HTTP_MULTIRANGE": "YES",
    "GDAL_HTTP_MERGE_CONSECUTIVE_RANGES": "YES",
}


def hansen_tile(lat: float, lon: float) -> str:
    top = math.ceil(lat / 10) * 10
    left = math.floor(lon / 10) * 10
    return f"{abs(top):02d}{'N' if top >= 0 else 'S'}_{abs(left):03d}{'E' if left >= 0 else 'W'}"


def _bbox(region: Region, half_m: float) -> tuple[float, float, float, float]:
    dlat = half_m / 111_320
    dlon = half_m / (111_320 * math.cos(math.radians(region.lat)))
    return region.lon - dlon, region.lat - dlat, region.lon + dlon, region.lat + dlat


def _search(catalog: pystac_client.Client, region: Region, window: str, tile: str | None, n: int) -> list:
    lon0, lat0, lon1, lat1 = _bbox(region, WINDOW_PX * 10 / 2)
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=[lon0, lat0, lon1, lat1],
        datetime=window,
        query={"eo:cloud_cover": {"lt": 70}},
        max_items=300,
    )
    items = list(search.items())
    if tile is None:
        # the MGRS tile that fully contains the window and has the most clear scenes
        counts: dict[str, int] = {}
        for it in items:
            counts[it.properties["s2:mgrs_tile"]] = counts.get(it.properties["s2:mgrs_tile"], 0) + 1
        tile = max(counts, key=lambda k: counts[k])
    items = [it for it in items if it.properties["s2:mgrs_tile"] == tile]
    items.sort(key=lambda it: it.properties["eo:cloud_cover"])
    return items[:n]


def _target_grid(region: Region, crs: str) -> tuple[rasterio.Affine, tuple[float, float, float, float]]:
    from pyproj import Transformer

    x, y = Transformer.from_crs("EPSG:4326", crs, always_xy=True).transform(region.lon, region.lat)
    half = WINDOW_PX * 10 / 2
    x0 = round((x - half) / 10) * 10
    y1 = round((y + half) / 10) * 10
    transform = rasterio.Affine(10.0, 0.0, x0, 0.0, -10.0, y1)
    return transform, (x0, y1 - WINDOW_PX * 10, x0 + WINDOW_PX * 10, y1)


def _read_asset(href: str, bounds: tuple[float, float, float, float], resampling: Resampling) -> np.ndarray:
    with rasterio.Env(**_GDAL_ENV), rasterio.open(href) as src:
        window = from_bounds(*bounds, src.transform)
        return src.read(
            1,
            window=window,
            out_shape=(WINDOW_PX, WINDOW_PX),
            resampling=resampling,
            boundless=True,
            fill_value=0,
        )


def _composite(items: list, bounds: tuple[float, float, float, float]) -> tuple[np.ndarray, np.ndarray, list]:
    stack = np.zeros((len(items), len(BANDS), WINDOW_PX, WINDOW_PX), dtype=np.float32)
    clear = np.zeros((len(items), WINDOW_PX, WINDOW_PX), dtype=bool)
    jobs = []
    for i, it in enumerate(items):
        for b, band in enumerate(BANDS):
            res = Resampling.bilinear if band == "B11" else Resampling.nearest
            jobs.append((i, b, it.assets[band].href, res))
        jobs.append((i, -1, it.assets["SCL"].href, Resampling.nearest))

    def run(job: tuple) -> tuple[int, int, np.ndarray]:
        i, b, href, res = job
        for attempt in range(3):
            try:
                return i, b, _read_asset(href, bounds, res)
            except Exception:
                if attempt == 2:
                    raise
                time.sleep(3)
        raise RuntimeError("unreachable")

    with ThreadPoolExecutor(8) as ex:
        for i, b, arr in ex.map(run, jobs):
            if b == -1:
                clear[i] = np.isin(arr, CLEAR_SCL)
            else:
                stack[i, b] = arr.astype(np.float32)
    meta = []
    for i, it in enumerate(items):
        baseline = float(it.properties.get("s2:processing_baseline", "0") or 0)
        offset = 1000.0 if baseline >= 4.0 else 0.0  # BOA_ADD_OFFSET introduced with baseline 04.00
        stack[i] = np.clip((stack[i] - offset) / 10_000.0, 0, 1.5)
        meta.append({"id": it.id, "cloud": it.properties["eo:cloud_cover"], "baseline": baseline})
    # pixels at a swath edge (no data in any band) or with a bright blue band are not clear
    clear &= (stack > 0).all(axis=1) & (stack[:, 0] < CLOUD_BLUE)
    # drop hazy scenes: median blue over clear pixels well above the clearest scene's
    blue = np.array(
        [np.median(stack[i, 0][clear[i]]) if clear[i].mean() > 0.05 else np.inf for i in range(len(items))]
    )
    if np.isfinite(blue).any():
        ref = np.nanmin(blue[np.isfinite(blue)])
        for i in range(len(items)):
            if not np.isfinite(blue[i]) or blue[i] > ref * 1.6:
                clear[i] = False
    # Minimum-blue compositing: among clear observations take, per pixel, the one with the lowest blue
    # reflectance. Haze and cloud remnants raise blue, so this suppresses them (shadows are already
    # excluded by the scene classification).
    blue_masked = np.where(clear, stack[:, 0], np.inf)
    pick = np.argmin(blue_masked, axis=0)
    comp = np.take_along_axis(stack, pick[None, None], axis=0)[0]
    n_clear = clear.sum(axis=0).astype(np.uint8)
    comp[:, n_clear == 0] = 0.0
    return (comp * 10_000).astype(np.uint16), n_clear, meta


def _reproject_layer(href: str, region: Region, transform: rasterio.Affine, crs: str) -> np.ndarray:
    lon0, lat0, lon1, lat1 = _bbox(region, WINDOW_PX * 10 / 2 + 600)
    with rasterio.Env(**_GDAL_ENV), rasterio.open(href) as src:
        window = from_bounds(lon0, lat0, lon1, lat1, src.transform).round_offsets().round_lengths()
        data = src.read(1, window=window)
        src_transform = src.window_transform(window)
        src_crs = src.crs
    out = np.zeros((WINDOW_PX, WINDOW_PX), dtype=np.uint8)
    reproject(
        data,
        out,
        src_transform=src_transform,
        src_crs=src_crs,
        dst_transform=transform,
        dst_crs=crs,
        resampling=Resampling.nearest,
    )
    return out


def fetch_region(catalog: pystac_client.Client, region: Region, n_scenes: int) -> Path:
    out = OUT / f"{region.key}.npz"
    if out.exists():
        print(f"[skip] {region.key}")
        return out
    t0 = time.time()
    base_items = _search(catalog, region, BASELINE_WINDOW, None, n_scenes)
    if not base_items:
        raise RuntimeError(f"no baseline scenes for {region.key}")
    tile = base_items[0].properties["s2:mgrs_tile"]
    cur_items = _search(catalog, region, CURRENT_WINDOW, tile, n_scenes)
    with rasterio.open(base_items[0].assets["B04"].href) as src:
        crs = src.crs.to_string()
    transform, bounds = _target_grid(region, crs)
    s2_base, clear_base, meta_base = _composite(base_items, bounds)
    s2_cur, clear_cur, meta_cur = _composite(cur_items, bounds)
    tile_id = hansen_tile(region.lat, region.lon)
    lossyear = _reproject_layer(
        f"{HANSEN_BASE}/Hansen_{HANSEN_VERSION}_lossyear_{tile_id}.tif", region, transform, crs
    )
    treecover = _reproject_layer(
        f"{HANSEN_BASE}/Hansen_{HANSEN_VERSION}_treecover2000_{tile_id}.tif", region, transform, crs
    )
    jrc = _reproject_layer(JRC_GFC2020_COG, region, transform, crs)
    OUT.mkdir(parents=True, exist_ok=True)
    meta = {
        "region": region.key,
        "country": region.country,
        "commodity": region.commodity,
        "crs": crs,
        "transform": list(transform)[:6],
        "bounds_4326": transform_bounds(crs, "EPSG:4326", *bounds),
        "mgrs_tile": tile,
        "bands": list(BANDS),
        "baseline_scenes": meta_base,
        "current_scenes": meta_cur,
        "hansen": HANSEN_VERSION,
        "jrc": JRC_VERSION,
    }
    np.savez_compressed(
        out,
        s2_baseline=s2_base,
        s2_current=s2_cur,
        clear_baseline=clear_base,
        clear_current=clear_cur,
        lossyear=lossyear,
        treecover2000=treecover,
        jrc_forest2020=jrc,
        meta=np.array(json.dumps(meta)),
    )
    print(
        f"[ok] {region.key} tile={tile} scenes={len(base_items)}/{len(cur_items)} "
        f"loss21+={(lossyear >= 21).mean():.3f} jrc={jrc.mean():.2f} in {time.time() - t0:.0f}s"
    )
    return out


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--regions", default="", help="comma-separated region keys (default: all)")
    ap.add_argument("--scenes", type=int, default=8, help="scenes per epoch for the clear-sky composite")
    ap.add_argument("--set", default="demo", choices=["demo", "full"], help="region set (full = GPU scale)")
    args = ap.parse_args()
    wanted = {k for k in args.regions.split(",") if k}
    catalog = pystac_client.Client.open(STAC_URL, modifier=planetary_computer.sign_inplace)
    for region in regions_for(args.set):
        if wanted and region.key not in wanted:
            continue
        try:
            fetch_region(catalog, region, args.scenes)
        except Exception as exc:  # keep going; a region can be retried later
            print(f"[fail] {region.key}: {type(exc).__name__}: {exc}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/plotline/ml/plotline_ml/patches.py
"""Cut each region window into 32 x 32 patches (10.24 ha "plots") and attach plot-level labels.

Labels come only from the official/public forest datasets and are used for evaluation, never training:
- ``forest2020``: share of the plot that the EU JRC GFC2020 map marks as forest on 31 Dec 2020;
- ``loss_after``: share of the plot that was forest in 2020 (JRC) *and* has Hansen loss in 2021-2025;
- ``loss_before``: share with Hansen loss in 2001-2020 (context only).
"""

from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path

import numpy as np

from plotline_ml.regions import ALL_REGIONS, PATCH_PX

ROOT = Path(__file__).resolve().parents[2]
REGION_DIR = ROOT / "data" / "eo" / "regions"


@dataclass
class PatchSet:
    baseline: np.ndarray  # (N, 5, 32, 32) float32 reflectance
    current: np.ndarray  # (N, 5, 32, 32)
    clear_baseline: np.ndarray  # (N,) share of pixels with at least one clear observation
    clear_current: np.ndarray
    forest2020: np.ndarray  # (N,)
    treecover2000: np.ndarray  # (N,) mean percent
    loss_after: np.ndarray  # (N,)
    loss_before: np.ndarray  # (N,)
    region: np.ndarray  # (N,) region key
    row: np.ndarray
    col: np.ndarray

    def __len__(self) -> int:
        return len(self.region)

    def subset(self, mask: np.ndarray) -> PatchSet:
        return PatchSet(**{k: getattr(self, k)[mask] for k in self.__dataclass_fields__})


def _tile(a: np.ndarray) -> np.ndarray:
    """(..., H, W) -> (n, ..., P, P) in row-major patch order."""
    p = PATCH_PX
    h, w = a.shape[-2:]
    lead = a.shape[:-2]
    a = a.reshape(*lead, h // p, p, w // p, p)
    a = np.moveaxis(a, (-4, -2), (0, 1))  # (H/p, W/p, ..., p, p)
    return a.reshape(-1, *lead, p, p)


def load_patches(region_keys: list[str] | None = None) -> PatchSet:
    parts: dict[str, list[np.ndarray]] = {k: [] for k in PatchSet.__dataclass_fields__}
    for region in ALL_REGIONS:  # whichever regions have been fetched
        if region_keys and region.key not in region_keys:
            continue
        path = REGION_DIR / f"{region.key}.npz"
        if not path.exists():
            continue
        d = np.load(path)
        base = _tile(d["s2_baseline"].astype(np.float32) / 10_000)
        cur = _tile(d["s2_current"].astype(np.float32) / 10_000)
        jrc = _tile(d["jrc_forest2020"] == 1)
        loss = d["lossyear"]
        after = _tile((loss >= 21) & (d["jrc_forest2020"] == 1))
        before = _tile((loss >= 1) & (loss <= 20))
        n = base.shape[0]
        grid = int(np.sqrt(n))
        parts["baseline"].append(base)
        parts["current"].append(cur)
        parts["clear_baseline"].append(_tile(d["clear_baseline"] > 0).mean(axis=(1, 2)))
        parts["clear_current"].append(_tile(d["clear_current"] > 0).mean(axis=(1, 2)))
        parts["forest2020"].append(jrc.mean(axis=(1, 2)))
        parts["treecover2000"].append(_tile(d["treecover2000"].astype(np.float32)).mean(axis=(1, 2)))
        parts["loss_after"].append(after.mean(axis=(1, 2)))
        parts["loss_before"].append(before.mean(axis=(1, 2)))
        parts["region"].append(np.array([region.key] * n))
        parts["row"].append(np.repeat(np.arange(grid), grid))
        parts["col"].append(np.tile(np.arange(grid), grid))
    if not parts["region"]:
        raise FileNotFoundError(f"No region files in {REGION_DIR}. Run `make ml-data` first.")
    return PatchSet(**{k: np.concatenate(v) for k, v in parts.items()})


def region_meta(key: str) -> dict:
    return json.loads(str(np.load(REGION_DIR / f"{key}.npz")["meta"]))

In [ ]:
%%writefile /kaggle/working/plotline/ml/plotline_ml/model.py
"""Tiny convolutional encoder for 32 x 32 Sentinel-2 patches (5 bands), ~0.45 M parameters.

The architecture is deliberately plain (3x3 conv + batch norm + ReLU, global average pooling, linear head)
so the trained network can be exported to a handful of NumPy arrays and run inside the API without any
deep-learning runtime (see ``export.py`` and ``backend/app/ml/encoder.py``).
"""

from __future__ import annotations

import torch
from torch import nn

CHANNELS: tuple[int, ...] = (32, 64, 64, 128, 128, 128)
STRIDES: tuple[int, ...] = (1, 2, 1, 2, 1, 2)
EMBED_DIM = 128
PROJ_DIM = 64


class TinyEncoder(nn.Module):
    def __init__(self, in_ch: int = 5, embed_dim: int = EMBED_DIM) -> None:
        super().__init__()
        layers: list[nn.Module] = []
        c = in_ch
        for out, stride in zip(CHANNELS, STRIDES, strict=True):
            layers += [
                nn.Conv2d(c, out, 3, stride, 1, bias=False),
                nn.BatchNorm2d(out),
                nn.ReLU(inplace=True),
            ]
            c = out
        self.features = nn.Sequential(*layers)
        self.head = nn.Linear(c, embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.features(x).mean(dim=(2, 3)))


class Projector(nn.Module):
    """Training-only projection head; SIGReg and the invariance term act on its output."""

    def __init__(self, embed_dim: int = EMBED_DIM, proj_dim: int = PROJ_DIM) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, 256), nn.BatchNorm1d(256), nn.ReLU(inplace=True), nn.Linear(256, proj_dim)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def count_params(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters())

In [ ]:
%%writefile /kaggle/working/plotline/ml/plotline_ml/npencoder.py
"""NumPy-only forward pass of the exported encoder (batch norm folded into the convolutions).

This exact file is vendored as ``backend/app/ml/encoder.py`` so the API computes embeddings without a
deep-learning runtime. ``tests/test_export.py`` checks it against the PyTorch model.
"""

from __future__ import annotations

import hashlib
import json
from dataclasses import dataclass
from pathlib import Path

import numpy as np
from numpy.lib.stride_tricks import sliding_window_view


@dataclass(frozen=True)
class EncoderWeights:
    convs: tuple[tuple[np.ndarray, np.ndarray, int], ...]  # (weight (O,C,3,3), bias (O,), stride)
    head_w: np.ndarray  # (D, C)
    head_b: np.ndarray  # (D,)
    band_mean: np.ndarray  # (C_in,)
    band_std: np.ndarray  # (C_in,)
    meta: dict
    sha256: str

    @property
    def embed_dim(self) -> int:
        return int(self.head_w.shape[0])


def load_weights(path: str | Path) -> EncoderWeights:
    path = Path(path)
    raw = path.read_bytes()
    d = np.load(path, allow_pickle=False)
    n = int(d["n_convs"])
    convs = tuple(
        (d[f"conv{i}_w"].astype(np.float32), d[f"conv{i}_b"].astype(np.float32), int(d[f"conv{i}_stride"]))
        for i in range(n)
    )
    return EncoderWeights(
        convs=convs,
        head_w=d["head_w"].astype(np.float32),
        head_b=d["head_b"].astype(np.float32),
        band_mean=d["band_mean"].astype(np.float32),
        band_std=d["band_std"].astype(np.float32),
        meta=json.loads(str(d["meta"])),
        sha256=hashlib.sha256(raw).hexdigest(),
    )


def _conv3x3(x: np.ndarray, w: np.ndarray, b: np.ndarray, stride: int) -> np.ndarray:
    xp = np.pad(x, ((0, 0), (0, 0), (1, 1), (1, 1)))
    win = sliding_window_view(xp, (3, 3), axis=(2, 3))[:, :, ::stride, ::stride]  # (N,C,Ho,Wo,3,3)
    n, c, ho, wo = win.shape[:4]
    cols = win.transpose(0, 2, 3, 1, 4, 5).reshape(n * ho * wo, c * 9)
    out = cols @ w.reshape(w.shape[0], -1).T + b
    return out.reshape(n, ho, wo, -1).transpose(0, 3, 1, 2)


def embed(weights: EncoderWeights, patches: np.ndarray, batch: int = 64) -> np.ndarray:
    """patches: (N, C, 32, 32) surface reflectance (0-1). Returns (N, D) float32 embeddings."""
    x_all = np.asarray(patches, dtype=np.float32)
    if x_all.ndim == 3:
        x_all = x_all[None]
    outs = []
    for start in range(0, len(x_all), batch):
        x = (x_all[start : start + batch] - weights.band_mean[None, :, None, None]) / weights.band_std[
            None, :, None, None
        ]
        for w, b, stride in weights.convs:
            x = np.maximum(_conv3x3(x, w, b, stride), 0.0)
        outs.append(x.mean(axis=(2, 3)) @ weights.head_w.T + weights.head_b)
    return np.concatenate(outs).astype(np.float32) if outs else np.zeros((0, weights.embed_dim), np.float32)


def cosine_distance(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Row-wise 1 - cosine similarity, in [0, 2]."""
    a = a / (np.linalg.norm(a, axis=-1, keepdims=True) + 1e-8)
    b = b / (np.linalg.norm(b, axis=-1, keepdims=True) + 1e-8)
    return 1.0 - np.sum(a * b, axis=-1)

In [ ]:
%%writefile /kaggle/working/plotline/ml/plotline_ml/train.py
"""Self-supervised training of the tiny encoder with LeJEPA (SIGReg + view invariance), CPU-friendly.

    python -m plotline_ml.train [--steps 1500] [--batch 128] [--views 4] [--lamb 0.05]

LeJEPA objective (Balestriero & LeCun, 2025): for V augmented views of each patch,
    loss = (1 - lambda) * invariance + lambda * SIGReg
where invariance pulls every view's projection towards the mean over views and SIGReg (Sketched
Isotropic Gaussian Regularisation, ``lejepa.multivariate.SlicingUnivariateTest`` over an Epps-Pulley
univariate test) pushes the projected batch towards an isotropic Gaussian, which prevents collapse
without teacher networks, stop-gradients or schedules of heuristics.

Augmentations model what should NOT count as land change: viewing geometry (crops, flips, 90-degree
rotations), sensor/illumination gain, atmospheric haze (a blue-weighted additive offset with contrast
loss) and noise. No labels are used. Regions in ``HELD_OUT`` are excluded from training entirely.
"""

from __future__ import annotations

import argparse
import json
import math
import time
from pathlib import Path

import lejepa
import numpy as np
import torch
import torch.nn.functional as F

from plotline_ml.model import Projector, TinyEncoder, count_params
from plotline_ml.patches import load_patches

ROOT = Path(__file__).resolve().parents[2]
ARTIFACTS = ROOT / "ml" / "artifacts"
CHECKPOINT = ROOT / "data" / "ml" / "encoder.pt"
# Two regions never seen in training, used for the held-out evaluation. ci_soubre is not a candidate: the
# JRC GFC2020 map shows no forest there, so it has no eligible evaluation plots.
HELD_OUT = ("pe_madre_de_dios", "gh_western_north")
HAZE_PROFILE = torch.tensor([1.0, 0.8, 0.6, 0.3, 0.15])


def augment(x: torch.Tensor, g: torch.Generator) -> torch.Tensor:
    """x: (B, C, 32, 32) reflectance -> one augmented view, same shape."""
    b, dev = x.shape[0], x.device  # random draws stay on the CPU generator (same stream on any device)
    scale = torch.empty(b).uniform_(0.55, 1.0, generator=g)
    tx = (torch.rand(b, generator=g) * 2 - 1) * (1 - scale)
    ty = (torch.rand(b, generator=g) * 2 - 1) * (1 - scale)
    k = torch.randint(0, 4, (b,), generator=g).float() * (math.pi / 2)
    flip = torch.where(torch.rand(b, generator=g) < 0.5, -1.0, 1.0)
    cos, sin = torch.cos(k) * scale, torch.sin(k) * scale
    theta = torch.stack(
        [torch.stack([cos * flip, -sin, tx], 1), torch.stack([sin * flip, cos, ty], 1)], 1
    ).to(dev)  # (B, 2, 3)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    v = F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)
    gain = torch.empty(b, x.shape[1], 1, 1).uniform_(0.88, 1.12, generator=g).to(dev)
    haze = (
        torch.empty(b, 1, 1, 1).uniform_(0.0, 0.035, generator=g)
        * HAZE_PROFILE[: x.shape[1]].view(1, -1, 1, 1)
    ).to(dev)
    contrast = torch.empty(b, 1, 1, 1).uniform_(0.0, 0.2, generator=g).to(dev)
    mean = v.mean(dim=(2, 3), keepdim=True)
    v = (v * (1 - contrast) + mean * contrast) * gain + haze
    return v + torch.randn(v.shape, generator=g).to(dev) * 0.004


def normalise(x: torch.Tensor, mean: torch.Tensor, std: torch.Tensor) -> torch.Tensor:
    return (x - mean.view(1, -1, 1, 1)) / std.view(1, -1, 1, 1)


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--steps", type=int, default=1500)
    ap.add_argument("--batch", type=int, default=128)
    ap.add_argument("--views", type=int, default=4)
    ap.add_argument("--lamb", type=float, default=0.05)
    ap.add_argument("--lr", type=float, default=2e-3)
    ap.add_argument("--slices", type=int, default=256)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--threads", type=int, default=4)
    ap.add_argument("--device", default="auto", help="cpu, cuda or auto (CUDA when available)")
    args = ap.parse_args()
    device = torch.device(
        "cuda"
        if args.device == "auto" and torch.cuda.is_available()
        else ("cpu" if args.device == "auto" else args.device)
    )

    torch.manual_seed(args.seed)
    torch.set_num_threads(args.threads)
    g = torch.Generator().manual_seed(args.seed)

    ps = load_patches()
    train_mask = ~np.isin(ps.region, HELD_OUT)
    imgs = []
    for arr, clear in ((ps.baseline, ps.clear_baseline), (ps.current, ps.clear_current)):
        keep = train_mask & (clear >= 0.9)
        imgs.append(arr[keep])
    data = torch.from_numpy(np.concatenate(imgs)).float()
    band_mean = data.mean(dim=(0, 2, 3))
    band_std = data.std(dim=(0, 2, 3)) + 1e-6
    print(f"training images: {len(data)} from {len(set(ps.region[train_mask]))} regions")

    data, band_mean, band_std = data.to(device), band_mean.to(device), band_std.to(device)
    enc, proj = TinyEncoder(in_ch=data.shape[1]).to(device), Projector().to(device)
    print(f"encoder parameters: {count_params(enc):,} (+ projector {count_params(proj):,}, training only)")
    sigreg = lejepa.multivariate.SlicingUnivariateTest(
        univariate_test=lejepa.univariate.EppsPulley(n_points=17), num_slices=args.slices
    ).to(device)
    params = list(enc.parameters()) + list(proj.parameters())
    opt = torch.optim.AdamW(params, lr=args.lr, weight_decay=5e-4)
    warmup = max(args.steps // 15, 1)
    sched = torch.optim.lr_scheduler.SequentialLR(
        opt,
        [
            torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.01, total_iters=warmup),
            torch.optim.lr_scheduler.CosineAnnealingLR(
                opt, T_max=args.steps - warmup, eta_min=args.lr / 1000
            ),
        ],
        milestones=[warmup],
    )
    log: list[dict] = []
    t0 = time.time()
    enc.train()
    proj.train()
    for step in range(1, args.steps + 1):
        idx = torch.randint(0, len(data), (args.batch,), generator=g)
        x = data[idx.to(device)]
        views = torch.cat([normalise(augment(x, g), band_mean, band_std) for _ in range(args.views)])
        z = proj(enc(views)).view(args.views, args.batch, -1)  # (V, B, D)
        inv = (z.mean(0, keepdim=True) - z).square().mean()
        sig = sigreg(z)
        loss = (1 - args.lamb) * inv + args.lamb * sig
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        sched.step()
        if step % 50 == 0 or step == 1:
            rec = {
                "step": step,
                "loss": round(loss.item(), 5),
                "invariance": round(inv.item(), 5),
                "sigreg": round(sig.item(), 5),
                "seconds": round(time.time() - t0, 1),
            }
            log.append(rec)
            print(json.dumps(rec), flush=True)

    CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "encoder": {k: v.cpu() for k, v in enc.state_dict().items()},
            "band_mean": band_mean.cpu(),
            "band_std": band_std.cpu(),
            "args": vars(args),
            "held_out": HELD_OUT,
            "train_images": len(data),
            "train_seconds": round(time.time() - t0, 1),
        },
        CHECKPOINT,
    )
    ARTIFACTS.mkdir(parents=True, exist_ok=True)
    (ARTIFACTS / "train_log.json").write_text(
        json.dumps(
            {
                "args": vars(args),
                "held_out_regions": list(HELD_OUT),
                "train_images": len(data),
                "encoder_parameters": count_params(enc),
                "train_seconds": round(time.time() - t0, 1),
                "device": str(device),
                "curve": log,
            },
            indent=2,
        )
    )
    print(f"saved {CHECKPOINT} in {time.time() - t0:.0f}s")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/plotline/ml/plotline_ml/export.py
"""Export the trained encoder to a small NumPy archive (batch norm folded into conv weights).

    python -m plotline_ml.export

Writes ``ml/artifacts/encoder.npz`` (weights, band statistics, metadata) and ``ml/artifacts/reference.npz``
(a few input patches with their expected embeddings) so the API can verify its NumPy forward pass.
"""

from __future__ import annotations

import json
from datetime import UTC, datetime

import numpy as np
import torch

from plotline_ml.model import CHANNELS, STRIDES, TinyEncoder, count_params
from plotline_ml.npencoder import embed, load_weights
from plotline_ml.train import ARTIFACTS, CHECKPOINT, normalise

ENCODER_VERSION = "plotline-lejepa-s2-v1"


def fold(enc: TinyEncoder) -> dict[str, np.ndarray]:
    out: dict[str, np.ndarray] = {}
    mods = list(enc.features)
    i = 0
    for k in range(0, len(mods), 3):
        conv, bn = mods[k], mods[k + 1]
        scale = bn.weight.detach() / torch.sqrt(bn.running_var + bn.eps)
        out[f"conv{i}_w"] = (conv.weight.detach() * scale.view(-1, 1, 1, 1)).numpy().astype(np.float32)
        out[f"conv{i}_b"] = (bn.bias.detach() - bn.running_mean * scale).numpy().astype(np.float32)
        out[f"conv{i}_stride"] = np.array(conv.stride[0])
        i += 1
    out["n_convs"] = np.array(i)
    out["head_w"] = enc.head.weight.detach().numpy().astype(np.float32)
    out["head_b"] = enc.head.bias.detach().numpy().astype(np.float32)
    return out


def main() -> None:
    ck = torch.load(CHECKPOINT, weights_only=False, map_location="cpu")
    enc = TinyEncoder(in_ch=len(ck["band_mean"]))
    enc.load_state_dict(ck["encoder"])
    enc.eval()
    arrays = fold(enc)
    meta = {
        "version": ENCODER_VERSION,
        "architecture": {
            "channels": list(CHANNELS),
            "strides": list(STRIDES),
            "embed_dim": enc.head.out_features,
        },
        "parameters": count_params(enc),
        "bands": ["B02", "B03", "B04", "B08", "B11"],
        "patch_px": 32,
        "gsd_m": 10,
        "objective": "LeJEPA: (1-lambda)*invariance + lambda*SIGReg (Epps-Pulley, sliced)",
        "train_args": ck["args"],
        "held_out_regions": list(ck["held_out"]),
        "train_images": ck["train_images"],
        "train_seconds": ck["train_seconds"],
        "exported_at": datetime.now(UTC).isoformat(timespec="seconds"),
    }
    ARTIFACTS.mkdir(parents=True, exist_ok=True)
    path = ARTIFACTS / "encoder.npz"
    np.savez_compressed(
        path,
        **arrays,
        band_mean=ck["band_mean"].numpy().astype(np.float32),
        band_std=ck["band_std"].numpy().astype(np.float32),
        meta=np.array(json.dumps(meta)),
    )
    weights = load_weights(path)
    rng = np.random.default_rng(7)
    x = (rng.random((6, len(ck["band_mean"]), 32, 32)) * 0.3).astype(np.float32)
    with torch.no_grad():
        ref = enc(normalise(torch.from_numpy(x), ck["band_mean"], ck["band_std"])).numpy()
    got = embed(weights, x)
    err = float(np.abs(got - ref).max())
    np.savez_compressed(ARTIFACTS / "reference.npz", inputs=x, embeddings=ref)
    print(
        f"exported {path} ({path.stat().st_size / 1024:.0f} KB, sha256 {weights.sha256[:12]}), "
        f"max |numpy - torch| = {err:.2e}"
    )
    if err > 1e-3:
        raise SystemExit("NumPy forward pass does not match PyTorch")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/plotline/ml/plotline_ml/evaluate.py
"""Evaluate the change signal against Hansen forest-loss labels at plot level.

    python -m plotline_ml.evaluate

Plots are 32 x 32 patches (10.24 ha). A plot is *eligible* when the EU JRC GFC2020 map shows at least
30 % forest on 31 Dec 2020 and both composites are at least 90 % cloud-free. Among eligible plots:
- positive: >= 5 % of the plot (about 0.5 ha, the FAO minimum forest area) was forest in 2020 and has
  Hansen loss in 2021-2025;
- negative: no such loss at all;
- plots with 0-5 % loss are reported but left out of the binary metrics (ambiguous at 30 m).

Scores compared (higher = more change):
- ``lejepa``: cosine distance between the 2020 and current embeddings of the trained encoder;
- ``random_init``: the same architecture with random weights (what self-supervision adds);
- ``ndvi_drop``: mean NDVI 2020 minus mean NDVI now (the classic hand-made index);
- ``nbr_drop``: same with the normalised burn ratio (NIR vs SWIR), sensitive to clearing.

Thresholds are calibrated on one half of the plots (stratified, fixed seed) by maximising F1 and the
reported precision / recall / F1 come from the other half. Results go to ``ml/artifacts/eval.json`` and
``ml/artifacts/EVAL.md``. Similar-plot search is scored as precision@5 of nearest neighbours sharing the
plot's 2020 land cover class (forest vs non-forest).
"""

from __future__ import annotations

import json

import numpy as np
import torch
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score

from plotline_ml.model import TinyEncoder
from plotline_ml.npencoder import cosine_distance, embed, load_weights
from plotline_ml.patches import PatchSet, load_patches
from plotline_ml.train import ARTIFACTS, HELD_OUT, normalise

POS_LOSS = 0.05
MIN_FOREST = 0.30
MIN_CLEAR = 0.90
MAX_FALSE_ALARM = 0.10


def _ndvi(x: np.ndarray) -> np.ndarray:
    nir, red = x[:, 3], x[:, 2]
    return ((nir - red) / (nir + red + 1e-6)).mean(axis=(1, 2))


def _nbr(x: np.ndarray) -> np.ndarray:
    nir, swir = x[:, 3], x[:, 4]
    return ((nir - swir) / (nir + swir + 1e-6)).mean(axis=(1, 2))


def _random_init_scores(ps: PatchSet, band_mean: np.ndarray, band_std: np.ndarray) -> np.ndarray:
    torch.manual_seed(123)
    enc = TinyEncoder(in_ch=ps.baseline.shape[1]).eval()
    m, s = torch.from_numpy(band_mean), torch.from_numpy(band_std)
    with torch.no_grad():
        a = enc(normalise(torch.from_numpy(ps.baseline), m, s)).numpy()
        b = enc(normalise(torch.from_numpy(ps.current), m, s)).numpy()
    return cosine_distance(a, b)


def _metrics(y: np.ndarray, score: np.ndarray, rng: np.random.Generator) -> dict:
    out: dict = {"n": len(y), "positives": int(y.sum())}
    if y.min() == y.max():
        return out
    out["roc_auc"] = round(float(roc_auc_score(y, score)), 4)
    out["average_precision"] = round(float(average_precision_score(y, score)), 4)
    # stratified calibration / test split
    idx_pos, idx_neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    rng.shuffle(idx_pos)
    rng.shuffle(idx_neg)
    cal = np.concatenate([idx_pos[: len(idx_pos) // 2], idx_neg[: len(idx_neg) // 2]])
    test = np.concatenate([idx_pos[len(idx_pos) // 2 :], idx_neg[len(idx_neg) // 2 :]])
    p, r, t = precision_recall_curve(y[cal], score[cal])
    f1 = 2 * p[:-1] * r[:-1] / np.maximum(p[:-1] + r[:-1], 1e-9)
    thr = float(t[int(np.argmax(f1))])
    pred = score[test] >= thr
    tp = int((pred & (y[test] == 1)).sum())
    fp = int((pred & (y[test] == 0)).sum())
    fn = int((~pred & (y[test] == 1)).sum())
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    out.update(
        threshold=round(thr, 5),
        precision=round(prec, 4),
        recall=round(rec, 4),
        f1=round(2 * prec * rec / max(prec + rec, 1e-9), 4),
        test_n=len(test),
    )
    # Operating point for review routing: at most 10 % of loss-free forest plots flagged (calibration half)
    thr10 = float(np.quantile(score[cal][y[cal] == 0], 1 - MAX_FALSE_ALARM))
    pred10 = score[test] >= thr10
    tp10 = int((pred10 & (y[test] == 1)).sum())
    fp10 = int((pred10 & (y[test] == 0)).sum())
    out["at_10pct_false_alarms"] = {
        "threshold": round(thr10, 5),
        "precision": round(tp10 / max(tp10 + fp10, 1), 4),
        "recall": round(tp10 / max(int((y[test] == 1).sum()), 1), 4),
        "false_alarm_rate": round(fp10 / max(int((y[test] == 0).sum()), 1), 4),
    }
    return out


def _similar_precision(emb: np.ndarray, forest: np.ndarray, k: int = 5, n_query: int = 1500) -> float:
    rng = np.random.default_rng(0)
    q = rng.choice(len(emb), size=min(n_query, len(emb)), replace=False)
    e = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-8)
    sims = e[q] @ e.T
    sims[np.arange(len(q)), q] = -np.inf
    nn = np.argsort(-sims, axis=1)[:, :k]
    return float((forest[nn] == forest[q][:, None]).mean())


def main() -> None:
    weights = load_weights(ARTIFACTS / "encoder.npz")
    ps = load_patches()
    ok = (ps.clear_baseline >= MIN_CLEAR) & (ps.clear_current >= MIN_CLEAR)
    eligible = ok & (ps.forest2020 >= MIN_FOREST)
    pos = eligible & (ps.loss_after >= POS_LOSS)
    neg = eligible & (ps.loss_after == 0)
    use = pos | neg
    print(
        f"plots: {len(ps)}, cloud-free: {ok.sum()}, forest-2020 eligible: {eligible.sum()}, "
        f"positives: {pos.sum()}, negatives: {neg.sum()}, ambiguous: {(eligible & ~use).sum()}"
    )

    e_base = embed(weights, ps.baseline)
    e_cur = embed(weights, ps.current)
    scores = {
        "lejepa": cosine_distance(e_base, e_cur),
        "random_init": _random_init_scores(ps, weights.band_mean, weights.band_std),
        "ndvi_drop": _ndvi(ps.baseline) - _ndvi(ps.current),
        "nbr_drop": _nbr(ps.baseline) - _nbr(ps.current),
    }
    y_all = pos.astype(int)
    report: dict = {
        "labels": {
            "source": "Hansen GFC-2025-v1.13 loss 2021-2025 within EU JRC GFC2020-V4 forest",
            "positive_rule": f"loss share >= {POS_LOSS:.0%} of a 10.24 ha plot",
            "eligibility": f"JRC forest share >= {MIN_FOREST:.0%}, both composites >= {MIN_CLEAR:.0%} clear",
        },
        "counts": {
            "plots": len(ps),
            "eligible": int(eligible.sum()),
            "positives": int(pos.sum()),
            "negatives": int(neg.sum()),
            "ambiguous_excluded": int((eligible & ~use).sum()),
        },
        "encoder": weights.meta,
        "encoder_sha256": weights.sha256,
        "all_regions": {},
        "held_out_regions": {},
        "per_region_auc": {},
    }
    held = np.isin(ps.region, HELD_OUT)
    for name, s in scores.items():
        report["all_regions"][name] = _metrics(y_all[use], s[use], np.random.default_rng(42))
        report["held_out_regions"][name] = _metrics(
            y_all[use & held], s[use & held], np.random.default_rng(42)
        )
    for region in sorted(set(ps.region)):
        m = use & (ps.region == region)
        if m.sum() and 0 < y_all[m].sum() < m.sum():
            report["per_region_auc"][region] = {
                name: round(float(roc_auc_score(y_all[m], s[m])), 3) for name, s in scores.items()
            } | {"n": int(m.sum()), "positives": int(y_all[m].sum())}
    forest_cls = (ps.forest2020 >= 0.5).astype(int)
    report["similar_plots_precision_at_5"] = {
        "lejepa": round(_similar_precision(e_base[ok], forest_cls[ok]), 4),
        "chance": round(float(max(forest_cls[ok].mean(), 1 - forest_cls[ok].mean())), 4),
    }
    # The operating point the API uses (calibrated on all regions): the land-change signal routes plots to a
    # reviewer, so it is set for at most 10 % false alarms on loss-free forest plots rather than for best F1.
    report["api_threshold"] = (
        report["all_regions"]["lejepa"].get("at_10pct_false_alarms", {}).get("threshold")
    )
    report["api_threshold_rule"] = (
        "10 % false alarms on loss-free 2020-forest plots (all regions, calibration half)"
    )
    ARTIFACTS.mkdir(parents=True, exist_ok=True)
    (ARTIFACTS / "eval.json").write_text(json.dumps(report, indent=2))
    (ARTIFACTS / "EVAL.md").write_text(_markdown(report))
    print(_markdown(report))


def _reading(r: dict) -> list[str]:
    """A plain summary of what the numbers say, generated from them (so it never contradicts the tables)."""
    held, allr = r["held_out_regions"], r["all_regions"]
    best = max(("ndvi_drop", "nbr_drop"), key=lambda k: held[k].get("roc_auc", 0))
    best_all = max(("ndvi_drop", "nbr_drop"), key=lambda k: allr[k].get("roc_auc", 0))
    op = allr["lejepa"].get("at_10pct_false_alarms", {})
    beats_held = held["lejepa"].get("roc_auc", 0) > held[best].get("roc_auc", 0)
    beats_all = allr["lejepa"].get("roc_auc", 0) > allr[best_all].get("roc_auc", 0)
    ssl_note = (
        "Self-supervised training adds over the same untrained architecture."
        if held["lejepa"].get("roc_auc", 0) > held["random_init"].get("roc_auc", 0)
        else "Self-supervised training does not beat the untrained architecture here."
    )
    return [
        "## Reading the numbers",
        "",
        f"- On the regions held out of training the LeJEPA change score reaches ROC-AUC "
        f"{held['lejepa'].get('roc_auc')} ({'above' if beats_held else 'below'} the best spectral index, "
        f"{best} {held[best].get('roc_auc')}; random-init encoder "
        f"{held['random_init'].get('roc_auc')}). {ssl_note}",
        f"- Across all regions it reaches {allr['lejepa'].get('roc_auc')}, "
        f"{'above' if beats_all else 'below'} {best_all} ({allr[best_all].get('roc_auc')}): at demo scale "
        "(9 landscapes, 1,500 CPU steps) the score is not calibrated across landscapes.",
        f"- At the review operating point (at most 10 % false alarms) it recovers {op.get('recall')} of loss "
        f"plots across all regions with precision {op.get('precision')}. That is why the signal only routes "
        "plots to a reviewer and never decides a verdict; the official forest datasets do.",
        "- Held-out positives are the majority class (base rate above 50 %), so held-out precision and F1 "
        "are optimistic; ROC-AUC is the fairer comparison.",
        "",
    ]


def _markdown(r: dict) -> str:
    lines = [
        "# Land-change signal vs Hansen forest loss (plot level)",
        "",
        f"Labels: {r['labels']['source']}; positive = {r['labels']['positive_rule']}; "
        f"eligible = {r['labels']['eligibility']}.",
        "",
        f"Plots: {r['counts']['plots']} · eligible: {r['counts']['eligible']} · positives: "
        f"{r['counts']['positives']} · negatives: {r['counts']['negatives']} · ambiguous (excluded): "
        f"{r['counts']['ambiguous_excluded']}",
        "",
        *_reading(r),
    ]
    for scope, title in (
        ("all_regions", "All regions"),
        ("held_out_regions", "Held-out regions (never seen in training)"),
    ):
        lines += [
            f"## {title}",
            "",
            "| Score | ROC-AUC | Avg. precision | Precision | Recall | F1 | n (pos) |",
            "|---|---|---|---|---|---|---|",
        ]
        for name, m in r[scope].items():
            lines.append(
                f"| {name} | {m.get('roc_auc', '-')} | {m.get('average_precision', '-')} | "
                f"{m.get('precision', '-')} | {m.get('recall', '-')} | {m.get('f1', '-')} | "
                f"{m['n']} ({m['positives']}) |"
            )
        lines.append("")
        lines += [
            "At the review operating point (threshold set for at most 10 % false alarms on loss-free plots):",
            "",
            "| Score | Threshold | Precision | Recall | False-alarm rate |",
            "|---|---|---|---|---|",
        ]
        for name, m in r[scope].items():
            op = m.get("at_10pct_false_alarms")
            if op:
                lines.append(
                    f"| {name} | {op['threshold']} | {op['precision']} | {op['recall']} | "
                    f"{op['false_alarm_rate']} |"
                )
        lines.append("")
    lines += [
        "## ROC-AUC per region",
        "",
        "| Region | LeJEPA | Random init | NDVI drop | NBR drop | n (pos) |",
        "|---|---|---|---|---|---|",
    ]
    for region, m in r["per_region_auc"].items():
        lines.append(
            f"| {region} | {m['lejepa']} | {m['random_init']} | {m['ndvi_drop']} | {m['nbr_drop']} | "
            f"{m['n']} ({m['positives']}) |"
        )
    lines += [
        "",
        f"API operating threshold: {r['api_threshold']} (cosine distance; {r['api_threshold_rule']}).",
    ]
    sp = r["similar_plots_precision_at_5"]
    lines += [
        "",
        f"Similar-plot search precision@5 (same 2020 land cover): {sp['lejepa']} "
        f"(majority-class chance {sp['chance']}).",
        "",
    ]
    return "\n".join(lines)


if __name__ == "__main__":
    main()

## 1. Fetch the full region set (skips regions already on disk)

In [ ]:
!python -m plotline_ml.fetch --set full --scenes 8

## 2. Train with LeJEPA on the GPU

In [ ]:
!python -m plotline_ml.train --device cuda --steps 20000 --batch 512 --views 8 --lr 2e-3 --slices 512

## 3. Export the NumPy encoder (parity with PyTorch is checked on export)

In [ ]:
!python -m plotline_ml.export

## 4. Evaluate against Hansen loss on the held-out regions

In [ ]:
!python -m plotline_ml.evaluate

In [ ]:
from IPython.display import Markdown, display

display(Markdown((ROOT / "ml" / "artifacts" / "EVAL.md").read_text()))

In [ ]:
import shutil

shutil.make_archive("/kaggle/working/plotline_artifacts", "zip", ROOT / "ml" / "artifacts")
print("download /kaggle/working/plotline_artifacts.zip and unpack it into ml/artifacts/")